> **Sentetik veri.** Bu defter `python -m sentetik.uret` ile üretilen uydurma veriyle çalışır. Sayılar gerçek bir şirkete ait değildir ve gerçek projenin sonuçlarını temsil etmez; amaç yöntemi uçtan uca göstermektir. *(Synthetic data: numbers are not real results.)*

# Sinyal — 03 · Test (TEK SEFER)

**Bu defter test (2025-10-01) ve test2 (2026-01-01) kesimlerini açan tek yerdir.** Tanımlar: `docs/preregistration.md` v1.2 (kilitli). Seçimler geliştirme defterinde (02) donduruldu ve aşağıda sabit olarak yazılı; bu defterde **hiçbir seçim değiştirilmez**.

**Çalışma sırası:**
1. `KURU_CALISMA = True` (varsayılan): Test **açılmaz**. Bütün kod doğrulama kesimleriyle (2025-07 ve 2025-06 vekil hedef), eğitimin %30'u ve 50 bootstrap ile uçtan uca denenir. Amaç yalnızca hata yakalamak; sonuçlara bakılmaz.
2. Kuru çalışma hatasızsa `KURU_CALISMA = False` → kernel yeniden başlatılır → **Run All**. İlk metrik hesaplanırken `docs/sonuclar/03_TEST_ACILDI.txt` kilidi yazılır. Kilit varsa defter durur. Yeniden çalıştırma yalnızca kod hatası nedeniyle, günlüğe yazılarak yapılır (kilit dosyası elle silinir).

**Başarı ölçütleri (tanimlar v1.1 madde 3–4, 8, 10; v1.2 madde 2):**
- "Belirgin" B − A: testte capture@10 farkı ≥ 1 yüzde puan **ve** 1.000 tekrarlı müşteri düzeyi bootstrap %95 GA'sı 0'ı içermiyor **ve** 02'deki 4 katın en az 3'ünde fark testle aynı işarette.
- Bazları geçme: ana model, en iyi bazı hem PR-AUC'de hem capture@10'da geçmeli.
- Test2: B − A yönü testle aynı olmalı (destekler ya da sorgular, ana ölçütü değiştirmez).
- İkincil hipotez: B − A, tek siparişlilerde tekrar edenlerden büyük.
- ROC-AUC yalnızca raporlanır.

In [ ]:
import sys, time, json, datetime as dt
from pathlib import Path

KOK = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(KOK))

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

from src.veri import tablo_yukle
from src.ozellikler import (hazirla, model_a_ozellikleri, model_a_bayrak_ozellikleri,
                            model_b_ozellikleri)
from src.degerlendirme import (metrikler, alt_kumeler, bootstrap_fark, bootstrap_fark_farki,
                               capture_at_k, isaret_tutarli, hedef_egitim_indeksleri,
                               TEST_PLANI, ISARET_MIN_KAT, ETIKET_UFKU_GUN)
from src.modeller import baz_kural, baz_rfm, baz_bgnbd, egit, skorla

# ---------------- DONDURULMUŞ SEÇİMLER (02 sonrası, tanimlar v1.2) — DEĞİŞTİRME ----------------
ANA_MODEL = "lightgbm"
REFERANS_MODEL = "lojistik"
DAVRANIS_YONTEMI = "yuzdelik_aktif"
WEB_PAYI_GORELI = True
ESIK = 0.01            # "belirgin": capture@10 farkı ≥ 1 yüzde puan
BOOTSTRAP_N = 1000
TOHUM = 42

# ---------------- ÇALIŞMA KİPİ ----------------
KURU_CALISMA = True    # True: test AÇILMAZ (kod denemesi). Hatasızsa False → kernel yeniden başlat → Run All.

if KURU_CALISMA:
    PLAN = {"test": ("2025-07-01", ("egitim",)), "test2": ("2025-06-01", ("egitim",))}  # vekil hedefler
    BOOTSTRAP_N, EGITIM_ORANI, EK = 50, 0.3, "_KURU"
else:
    PLAN, EGITIM_ORANI, EK = TEST_PLANI, 1.0, ""

SONUC = KOK / "docs" / "sonuclar"; SONUC.mkdir(parents=True, exist_ok=True)
MODEL_DIZIN = KOK / "data" / "modeller_03"; MODEL_DIZIN.mkdir(parents=True, exist_ok=True)
KILIT = SONUC / "03_TEST_ACILDI.txt"
pd.set_option("display.float_format", "{:.4f}".format)

if not KURU_CALISMA and KILIT.exists():
    raise RuntimeError("Test daha önce açıldı:\n" + KILIT.read_text(encoding="utf-8")
                       + "\nYeniden çalıştırma yalnızca kod hatası nedeniyle ve gerekçesi yazılarak yapılır "
                         "(kilit dosyasını elle silin).")
print("KİP:", "KURU (test kapalı)" if KURU_CALISMA else "GERÇEK TEST — TEK SEFER")

## 1. Veri ve özellik setleri

In [ ]:
a = tablo_yukle("egitim_a")
b = tablo_yukle("egitim_b")
df = hazirla(a, b, davranis_yontemi=DAVRANIS_YONTEMI, web_payi_goreli=WEB_PAYI_GORELI)
df["y_etiket"] = df["y_etiket"].astype(int)

SETLER = {
    "A":        model_a_ozellikleri(df),
    "A+bayrak": model_a_bayrak_ozellikleri(df),
    "B":        model_b_ozellikleri(df),
}
assert [len(v) for v in SETLER.values()] == [24, 25, 38], {k: len(v) for k, v in SETLER.items()}
{k: len(v) for k, v in SETLER.items()}

## 2. Eğitim / hedef bölümleri ve sızdırmazlık kontrolleri

In [ ]:
BOLUMLER = {}
for ad, (hedef, kumeler) in PLAN.items():
    h = pd.Timestamp(hedef)
    egt = hedef_egitim_indeksleri(df, h, kumeler)
    hdf = df.index[df["kesim"] == h]
    assert len(egt) > 0 and len(hdf) > 0, ad
    assert (df.loc[egt, "kesim"] + pd.Timedelta(days=ETIKET_UFKU_GUN)).max() <= h, "etiket penceresi taşıyor"
    assert not df.loc[egt, "kesim"].eq(h).any(), "hedef kesim eğitimde"
    if not KURU_CALISMA:
        assert set(df.loc[hdf, "kume"]) == {ad}, "hedef kesimin kümesi yanlış"
    if EGITIM_ORANI < 1:
        egt = df.loc[egt].sample(frac=EGITIM_ORANI, random_state=TOHUM).index
    BOLUMLER[ad] = (h, egt, hdf)
    print(f"{ad:6s} hedef {h.date()} | eğitim kesimleri: "
          f"{sorted(df.loc[egt, 'kesim'].dt.strftime('%Y-%m').unique())} | n_eğitim={len(egt):,} n_hedef={len(hdf):,}")
# Beklenen (gerçek test): test ← 2024-10…2025-07 (10 kesim); test2 ← 2024-10…2025-10 (13 kesim)

## 3. Eğitim ve skorlama (metrik hesaplanmaz)

In [ ]:
SKOR = {}   # (bölüm, model, set) → skor dizisi
for ad, (h, egt, hdf) in BOLUMLER.items():
    e, d = df.loc[egt], df.loc[hdf]
    for set_adi, kol in SETLER.items():
        for m in [ANA_MODEL, REFERANS_MODEL]:
            t0 = time.time()
            model = egit(m, e, kol)
            SKOR[(ad, m, set_adi)] = skorla(m, model, d, kol)
            joblib.dump(model, MODEL_DIZIN / f"{ad}_{m}_{set_adi.replace('+', '_')}{EK}.joblib")
            print(f"{ad:6s} {m:9s} {set_adi:9s} {time.time() - t0:5.0f} sn")
    SKOR[(ad, "kural", "-")] = baz_kural(d).to_numpy()
    SKOR[(ad, "rfm", "-")] = baz_rfm(d).to_numpy()
    try:
        SKOR[(ad, "bgnbd", "-")] = baz_bgnbd(d).to_numpy()
    except Exception as hata:
        print("BG/NBD atlandı:", hata)
print("Skorlama tamam:", len(SKOR), "skor dizisi")

## 4. Metrikler — TEST BURADA AÇILIR (7 alt küme, ROC-AUC yalnızca rapor)

In [ ]:
if not KURU_CALISMA:
    KILIT.write_text(f"Test ilk kez açıldı: {dt.datetime.now().isoformat(timespec='seconds')}\n"
                     f"Seçimler: {ANA_MODEL}, {DAVRANIS_YONTEMI}, web_payi_goreli={WEB_PAYI_GORELI}\n",
                     encoding="utf-8")

OLASILIK = {ANA_MODEL, REFERANS_MODEL}
satirlar = []
for (ad, m, s), skor in SKOR.items():
    d = df.loc[BOLUMLER[ad][2]]
    y = d["y_etiket"].to_numpy()
    for ak, mask in alt_kumeler(d).items():
        satirlar.append({"bolum": ad, "model": m, "set": s, "alt_kume": ak,
                         **metrikler(y[mask], skor[mask], olasilik=m in OLASILIK)})
M = pd.DataFrame(satirlar)
M.to_csv(SONUC / f"03_metrikler{EK}.csv", index=False)
(M.query("alt_kume == 'tum'")
   .pivot_table(index=["model", "set"], columns="bolum",
                values=["pr_auc", "capture@10", "kesinlik@10", "brier", "roc_auc"]))

## 5. Ana ölçüt: B − A (bootstrap %95 GA, müşteri düzeyi)

In [ ]:
def pr_auc(y, s):
    return average_precision_score(y, s)

def fark(ad, m, ak, metrik_adi, metrik):
    d = df.loc[BOLUMLER[ad][2]]
    mask = alt_kumeler(d)[ak]
    y = d["y_etiket"].to_numpy()[mask]
    r = bootstrap_fark(y, SKOR[(ad, m, "A")][mask], SKOR[(ad, m, "B")][mask],
                       metrik=metrik, n=BOOTSTRAP_N, tohum=TOHUM)
    return {"bolum": ad, "model": m, "alt_kume": ak, "metrik": metrik_adi, "n": int(mask.sum()), **r}

t0 = time.time(); FARK = []
for ad in BOLUMLER:
    for ak in ["tum", "karar", "kapsam", "davranissiz", "tek_siparis", "tekrar", "uzak_180"]:
        FARK.append(fark(ad, ANA_MODEL, ak, "capture@10", capture_at_k))
    FARK.append(fark(ad, ANA_MODEL, "tum", "pr_auc", pr_auc))
    FARK.append(fark(ad, REFERANS_MODEL, "tum", "capture@10", capture_at_k))
    print(ad, "bootstrap tamam", round(time.time() - t0), "sn")
FD = pd.DataFrame(FARK)
FD.to_csv(SONUC / f"03_bootstrap{EK}.csv", index=False)
(FD.assign(**{c: FD[c] * 100 for c in ["fark", "alt_%2.5", "ust_%97.5"]})   # yüzde puan
   .round(2))

## 6. Karar tablosu (önceden yazılmış ölçütler)

In [ ]:
def satir(ad, ak="tum", metrik="capture@10", m=ANA_MODEL):
    return FD.query("bolum == @ad and alt_kume == @ak and metrik == @metrik and model == @m").iloc[0]

t, t2 = satir("test"), satir("test2")
kat = (pd.read_csv(SONUC / "02_gelisim.csv")
         .query("model == @ANA_MODEL and alt_kume == 'tum'")
         .pivot_table(index="kat", columns="set", values="capture@10"))
kat_farklari = (kat["B"] - kat["A"]).to_numpy()

tm = M.query("bolum == 'test' and alt_kume == 'tum'")
en_iyi_baz = tm[tm["model"].isin(["kural", "rfm", "bgnbd"])][["pr_auc", "capture@10"]].max()
ana_b = tm.query("model == @ANA_MODEL and set == 'B'").iloc[0]
ana_a = tm.query("model == @ANA_MODEL and set == 'A'").iloc[0]

K = pd.DataFrame([
    ("K1 capture@10 farkı ≥ 1 puan", f"{t.fark * 100:+.2f} puan", t.fark >= ESIK),
    ("K2 %95 GA 0'ı içermiyor", f"[{t['alt_%2.5'] * 100:+.2f}, {t['ust_%97.5'] * 100:+.2f}]",
     (t["alt_%2.5"] > 0) or (t["ust_%97.5"] < 0)),
    (f"K3 02 katlarının ≥ {ISARET_MIN_KAT}/4'ü testle aynı işaret",
     " ".join(f"{x * 100:+.2f}" for x in kat_farklari), isaret_tutarli(kat_farklari, test_farki=t.fark)),
    ("Baz geçme (B): PR-AUC ve capture@10", f"{ana_b.pr_auc:.4f}/{ana_b['capture@10']:.4f} vs "
     f"{en_iyi_baz.pr_auc:.4f}/{en_iyi_baz['capture@10']:.4f}",
     (ana_b.pr_auc > en_iyi_baz.pr_auc) and (ana_b["capture@10"] > en_iyi_baz["capture@10"])),
    ("Baz geçme (A): PR-AUC ve capture@10", f"{ana_a.pr_auc:.4f}/{ana_a['capture@10']:.4f}",
     (ana_a.pr_auc > en_iyi_baz.pr_auc) and (ana_a["capture@10"] > en_iyi_baz["capture@10"])),
    ("Test2: B − A yönü testle aynı", f"{t2.fark * 100:+.2f} puan", np.sign(t2.fark) == np.sign(t.fark)),
], columns=["olcut", "deger", "saglandi"])
BELIRGIN = bool(K.loc[:2, "saglandi"].all())
K.to_csv(SONUC / f"03_karar{EK}.csv", index=False)
print("SONUÇ: B − A", "BELİRGİN" if BELIRGIN else "belirgin DEĞİL", "(K1 ∧ K2 ∧ K3)")
K

## 7. İkincil hipotez: B − A tek siparişlilerde tekrar edenlerden büyük mü?

In [ ]:
IK = []
for ad in BOLUMLER:
    d = df.loc[BOLUMLER[ad][2]]
    mk = alt_kumeler(d)
    r = bootstrap_fark_farki(d["y_etiket"].to_numpy(), SKOR[(ad, ANA_MODEL, "A")], SKOR[(ad, ANA_MODEL, "B")],
                             mk["tek_siparis"], mk["tekrar"], n=BOOTSTRAP_N, tohum=TOHUM)
    IK.append({"bolum": ad, **r})
IK = pd.DataFrame(IK)
IK.to_csv(SONUC / f"03_ikincil_hipotez{EK}.csv", index=False)
(IK.set_index("bolum") * 100).round(2)   # yüzde puan: fark_alt1 = tek, fark_alt2 = tekrar

## 8. Skorları ve çalışma bilgisini kaydet (3. hafta: kalibrasyon, SHAP, aksiyon matrisi)

In [ ]:
KOLON = ["kesim", "musteri_id", "kume", "y_etiket", "y_siparis_90g", "y_ciro_90g",
         "gun_son_siparis", "siparis_12ay", "davranis_var_30g"]
for ad, (h, egt, hdf) in BOLUMLER.items():
    d = df.loc[hdf, KOLON].copy()
    for (a2, m, s), sk in SKOR.items():
        if a2 == ad:
            d[f"skor_{m}" if s == "-" else f"skor_{m}_{s.replace('+', '_')}"] = sk
    d.to_parquet(KOK / "data" / f"03_skorlar_{ad}{EK}.parquet", index=False)   # data/ gitignore'da

bilgi = {"zaman": dt.datetime.now().isoformat(timespec="seconds"), "kuru": KURU_CALISMA,
         "plan": {k: [v[0], list(v[1])] for k, v in PLAN.items()},
         "ana_model": ANA_MODEL, "referans_model": REFERANS_MODEL,
         "davranis_yontemi": DAVRANIS_YONTEMI, "web_payi_goreli": WEB_PAYI_GORELI,
         "bootstrap_n": BOOTSTRAP_N, "egitim_orani": EGITIM_ORANI,
         "ozellikler": SETLER, "belirgin": BELIRGIN}
(SONUC / f"03_calisma_bilgisi{EK}.json").write_text(json.dumps(bilgi, ensure_ascii=False, indent=1), encoding="utf-8")
print("Kaydedildi.")

## 9. Sonuç ve yorum (doldurulacak)
- Ana ölçüt (K1–K3): …
- Bazlar: …
- Test2: …
- İkincil hipotez: …
- Alt gruplar (karar, kapsam, davranışsız, uzak_180): …